# Team 21 - EuroSAT label-efficiency: RGB grid
Settings (right panel): **Accelerator = GPU**, **Internet = On**. Run cells top to bottom.
Results are appended to `results/runs_rgb.csv`; finished runs are skipped, so restarting is safe.

In [ ]:
# 1) get the code from GitHub (public repo)
!git clone https://github.com/kumarAyush24029/Eurosat-label-efficiency.git /kaggle/working/proj
%cd /kaggle/working/proj
!pip install -q timm tifffile imagecodecs

In [ ]:
# 2) download RGB EuroSAT (~90 MB, official Zenodo mirror)
!mkdir -p data && cd data && wget -q -O rgb.zip 'https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip?download=1' && unzip -q -n rgb.zip -d raw_rgb
!ls data/raw_rgb

In [ ]:
# 3) quick GPU sanity run (should take under a minute)
!python src/train.py --rgb_root data/raw_rgb/EuroSAT_RGB --splits data/splits.json --out results/smoke.csv --models cnn --fracs 0.01 --seeds 0 --epochs 2

## Main grid (CNN, ResNet-50 scratch/ImageNet, ViT-S ImageNet) x 6 label fractions x 3 seeds

In [ ]:
COMMON = "--rgb_root data/raw_rgb/EuroSAT_RGB --splits data/splits.json --out results/runs_rgb.csv --fracs 0.01 0.02 0.05 0.1 0.25 1.0 --seeds 0 1 2 --save_logits"
!python src/train.py $COMMON --models cnn --inits scratch --inputs rgb --epochs 40
!python src/train.py $COMMON --models resnet50 --inits imagenet scratch --inputs rgb --epochs 20
!python src/train.py $COMMON --models vit_s --inits imagenet --inputs rgb --epochs 20

In [ ]:
# 4) plots + bundle to download (Output tab -> results_rgb.zip)
!python src/analyze.py --csv results/runs_rgb.csv --out figures
# error analysis: confusion matrices, per-class F1, calibration (uses the saved predictions)
!python src/error_analysis.py --model resnet50 --init imagenet --input rgb --fracs 0.01 0.1 1.0
!python src/error_analysis.py --model resnet50 --init scratch --input rgb --fracs 0.01 0.1 1.0
!python src/error_analysis.py --model vit_s --init imagenet --input rgb --fracs 0.01 0.1 1.0
!python src/eda.py --rgb_root data/raw_rgb/EuroSAT_RGB --out figures
!zip -qr results_rgb.zip results figures && ls -la results_rgb.zip